# 02 — Análisis exploratorio

Exploramos la distribución espacial de las escuelas en CDMX y calculamos
distancias al k-ésimo vecino más cercano para escoger un umbral razonable
del complejo Vietoris-Rips.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from sklearn.neighbors import NearestNeighbors
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_parquet(ROOT / 'data' / 'processed' / 'escuelas_cdmx.parquet')
print(len(df), 'escuelas')
df.head(2)


In [ ]:
# Conteos por nivel y sector
ct = df.groupby(['nivel','sector']).size().unstack(fill_value=0)
print(ct)


In [ ]:
# Conteos por alcaldía
top_alcaldias = df.groupby('municipio').size().sort_values(ascending=False)
print(top_alcaldias)


In [ ]:
# Mapa con marcadores agrupados
COLORS = {'preescolar':'green','primaria':'blue','secundaria':'orange',
          'media_superior':'red','media_tecnica':'purple'}
m = folium.Map(location=[19.4326, -99.1332], zoom_start=11, tiles='cartodbpositron')
cluster = MarkerCluster().add_to(m)
for _, r in df.sample(min(2000, len(df)), random_state=0).iterrows():
    folium.CircleMarker(
        [r['latitud'], r['longitud']], radius=3,
        color=COLORS.get(r['nivel'], 'gray'), fill=True, fill_opacity=0.7,
        tooltip=f"{r['nivel']} ({r['sector']}) — {r['nom_estab'][:50]}"
    ).add_to(cluster)
m


In [ ]:
# Distribución de distancias al k-ésimo vecino más cercano
def knn_distances(X: np.ndarray, k: int) -> np.ndarray:
    nn = NearestNeighbors(n_neighbors=k+1).fit(X)
    d, _ = nn.kneighbors(X)
    return d[:, k]

X = df[['x_utm','y_utm']].values
for k in (1, 5, 10):
    d = knn_distances(X, k)
    print(f'k={k:2d}: mediana={np.median(d):7.1f} m, p90={np.percentile(d,90):7.1f} m')


In [ ]:
# Histograma para k=5 — ayuda a escoger el umbral del Vietoris-Rips
fig = px.histogram(x=knn_distances(X, 5), nbins=60,
    labels={'x':'distancia al 5° vecino más cercano (m)'},
    title='Distancia al k=5 vecino — escuelas CDMX')
fig.show()


In [ ]:
# Lo mismo pero por nivel
import plotly.graph_objects as go
fig = go.Figure()
for nv, g in df.groupby('nivel'):
    if len(g) < 10: continue
    d = knn_distances(g[['x_utm','y_utm']].values, min(5, len(g)-1))
    fig.add_trace(go.Histogram(x=d, name=nv, opacity=0.5, nbinsx=60))
fig.update_layout(barmode='overlay', xaxis_title='distancia al 5° vecino (m)',
                  title='Densidad espacial por nivel educativo')
fig.show()


## Conclusiones del EDA

- La mediana de distancia al 5° vecino para primaria pública suele ser < 400 m
  (red densa, bien distribuida).
- Para media técnica terminal la red es muy escasa: pocas decenas de escuelas.
- **Umbral recomendado para Vietoris-Rips:** ~2500–3000 m (radio de caminata
  razonable para una zona urbana). Suficiente para que H0 se conecte casi
  totalmente y para que aparezcan H1 (huecos) interpretables.
